In [ ]:
import json
import os
import pandas as pd

In [ ]:
# Get list of all files in the directory
files = os.listdir('./../Generation/Filtered_Output/')
jsonl_files = sorted([file for file in files if file.endswith('.jsonl') and 
                      (file.startswith('dataset_nl_prompt_best') or 
                       file.startswith('dataset_java_nl_prompt_best') or 
                       (file.startswith('github-dataset_nl_prompt_best') and 'java' not in file) or 
                       file.startswith('github-dataset_java_nl_prompt_best'))])
print(len(jsonl_files))

In [ ]:
def get_result(file_path):
    df = pd.read_csv(file_path)
    test_success = None
    test_vulnerability = None

    if 'TestName' in df.columns:
        # Format 1: TestName, Result, Message
        for index, row in df.iterrows():
            if 'correctness' in str(row['TestName']):
                test_success = row['Result']
            if 'vulnerability' in str(row['TestName']):
                test_vulnerability = row['Result']
    elif 'test' in df.columns:
        # Format 2: test, status, error_reason
        # 'Error' means the test could not run at all -> return None (unknown result)
        for index, row in df.iterrows():
            status = str(row['status']).lower()
            if status == 'pass':
                result = 'success'
            elif status in ('fail', 'failure'):
                result = 'failure'
            else:  # 'error' or unknown
                result = None
            test_name = str(row['test']).lower()
            if 'functionality' in test_name or 'correctness' in test_name:
                test_success = result
            if 'security' in test_name or 'vulnerability' in test_name:
                test_vulnerability = result

    return test_success, test_vulnerability


In [ ]:
GENERATION_DIR = './../Generation/Filtered_Output/'

for file_name in jsonl_files:
    # Determine results directory based on file type
    is_java = 'dataset_java' in file_name
    is_github = file_name.startswith('github-dataset')
    if is_java and is_github:
        TEST_MODELS_RESULTS_DIR = './TestModelsResults_GitHub_Java'
    elif is_java:
        TEST_MODELS_RESULTS_DIR = './TestModelsResults_Java'
    elif is_github:
        TEST_MODELS_RESULTS_DIR = './TestModelsResults_GitHub_Python'
    else:
        TEST_MODELS_RESULTS_DIR = './TestModelsResults_Python'

    # Extract temp
    try:
        temp = file_name.rsplit('_', 1)[1].replace('.jsonl', '')
    except:
        print(f"Skipping {file_name}, cannot parse temp")
        continue
        
    model_base_name = file_name.replace('.jsonl', '')
    print(f"Processing {file_name}...")

    with open(GENERATION_DIR + file_name, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f.readlines()]

    for i in range(len(data)):
        # Robust ID extraction
        original_id = data[i].get('id', 'unknown')
        if not original_id or original_id == 'unknown':
            continue
        
        ext = os.path.splitext(original_id)[1]
        item_id_no_ext = os.path.splitext(original_id)[0]
        
        technique = data[i].get('technique', 'Assertion')
        source = data[i].get('source', 'Author')
        
        # Clean item_id
        item_id = item_id_no_ext
        prefix = f"{technique}_{source}_"
        if item_id.startswith(prefix):
            item_id = item_id[len(prefix):]
            
        # Determine Language
        is_java_dataset = 'dataset_java' in file_name
        language = "Java" if is_java_dataset else "Python"
        
        generations = data[i].get('generations', {})
        
        if generations:
             for lang_key, code_list in generations.items():
                for idx, code_obj in enumerate(code_list):
                    if '_' in model_base_name:
                        parts = model_base_name.rsplit('_', 1)
                        dir_name = f"{parts[0]}_{lang_key}_{parts[1]}"
                    else:
                        dir_name = f"{model_base_name}_{lang_key}"
                        
                    parent_dir_name = f"{dir_name}_R{idx+1}"
                    result_filename = f"Model_{parent_dir_name}_{language}_{technique}_{item_id}_results.csv"
                    result_file = os.path.join(TEST_MODELS_RESULTS_DIR, f"temp_{temp}", result_filename)
                    
                    test_success = None
                    test_vulnerability = None
                    if os.path.exists(result_file):
                        test_success, test_vulnerability = get_result(result_file)
                        
                    # Update the object in memory
                    code_obj['test_success'] = test_success
                    code_obj['test_vulnerability'] = test_vulnerability
        else:
            # Fallback for 'output' field
            outputs = data[i].get('output', [])
            if not isinstance(outputs, list):
                outputs = [outputs]
                
            for j in range(len(outputs)):
                parent_dir_name = f"{model_base_name}_R{j+1}"
                result_filename = f"Model_{parent_dir_name}_{language}_{technique}_{item_id}_results.csv"
                result_file = os.path.join(TEST_MODELS_RESULTS_DIR, f"temp_{temp}", result_filename)
                
                test_success = None
                test_vulnerability = None
                if os.path.exists(result_file):
                    test_success, test_vulnerability = get_result(result_file)
                
                # Handle if output is list of dicts or strings
                if isinstance(data[i]['output'][j], dict):
                    data[i]['output'][j]['test_success'] = test_success
                    data[i]['output'][j]['test_vulnerability'] = test_vulnerability
            

    with open('./TestResults/' + file_name, 'w', encoding='utf-8') as f:
        for item in data:
            f.write("%s\n" % json.dumps(item, ensure_ascii=False))
